# 실험 4 첫 실행 로그 감사
기존 로그만 읽는다. 모델/API/임베딩 호출은 하지 않는다. 한 장면 × 세 조건 각 1회이며 자유 서술의 사실성과 전략 성과를 구분한다.


In [1]:
from pathlib import Path
from collections import Counter
import json
root = next(p for p in [Path.cwd(), *Path.cwd().parents] if (p / 'experiments/results').is_dir())
path = root / 'experiments/results/20261021T085658-4b14f08b'
def read(name):
    return [json.loads(line) for line in (path / (name + '.jsonl')).read_text().splitlines() if line.strip()]
trials = read('trials')
manifest = json.loads((path / 'manifest.json').read_text())
calls = read('model_calls')
requests = {r['call_id']: r for r in calls if r['event'] == 'request'}
responses = [r for r in calls if r['event'] == 'response']
assert len(trials) == 3 and all(t['valid'] for t in trials)
assert len(requests) == len(responses) == 3
assert all(r['usage']['actual_model'] == 'gpt-6.1-sol' for r in responses)
assert all(t['game_config']['openai_reasoning_effort'] == 'low' for t in trials)
assert manifest['source_hashes']['agents/prompts/poker_action_selection.txt'] == manifest['source_hashes']['agents/prompts/poker_action_selection_standard.txt']
assert len({r['system'] for r in requests.values()}) == 1
assert len({json.dumps(r['schema'], sort_keys=True) for r in requests.values()}) == 1
inputs = []
for r in requests.values():
    user = json.loads(r['user'])
    trial = next(t for t in trials if t['trial_id'] == r['trial_id'])
    evidence = user.pop('past_experiences')
    assert user['force_bet'] is True
    assert user['observation']['allowed_bets'] == ['call', 'raise', 'fold']
    assert 'A' not in user['observation']['visible_cards']
    public_path = path / 'data' / trial['trial_id'] / 'experiences' / (trial['game_id'] + '.json')
    public = json.loads(public_path.read_text())
    for match in evidence['matches']:
        parent = next(p for p in public if p['id'] == match['metadata']['experience_id'])
        speech = json.loads(match['text'])
        assert speech in parent['events'] and speech['player_id'] == 'B' and speech['target'] == 'A'
        assert match['round_context']['result'] == parent['result']
    inputs.append(user)
assert all(u == inputs[0] for u in inputs)
assert all(not any(r['memo_inputs'].values()) for r in read('contexts') if 'memo_inputs' in r)
print('Verified: three valid responses; same system/schema/current state; only supplied history differs.')
print('Requested models:', Counter(r['model'] for r in requests.values()))
print('Actual models:', Counter(r['usage']['actual_model'] for r in responses))
print('Total seconds:', round(manifest['total_duration_s'], 3))
print('Token totals:', {k: sum(r['usage'][k] for r in responses) for k in ('input_tokens', 'output_tokens', 'tokens')})
for t in trials:
    truth = t['ground_truth']
    assert all(r['status'] == 'accepted' for r in t['applied'])
    assert t['decision']['memory_updates'] == t['saved_memos']
    memos = path / 'data' / t['trial_id'] / 'memos/A'
    for scope, text in t['saved_memos'].items():
        file = (memos / 'opponents.txt' if scope == 'long' else memos / t['game_id'] / 'game.txt' if scope == 'game' else memos / t['game_id'] / 'rounds/000001.txt')
        assert file.read_text() == text
    ranks = [c['target_rank'] for c in truth['claims']]
    print(t['condition'], t['betting_signature'], 'historical A ranks', ranks, 'true/false', truth['matching_claims'], truth['nonmatching_claims'])
    print('Round memo:', t['saved_memos']['round'])
    print('Long memo:', t['saved_memos']['long'])
print('Manual audit: both history arms name the correct historical A cards and accurately connect claim truth to trust and chosen action.')
print('Limit: one response per condition; no estimate of reproducibility, retrieval performance, reused-memory effect, or win-rate benefit.')


Verified: three valid responses; same system/schema/current state; only supplied history differs.
Requested models: Counter({'openai:gpt-6.1-sol': 3})
Actual models: Counter({'gpt-6.1-sol': 3})
Total seconds: 21.93
Token totals: {'input_tokens': 8921, 'output_tokens': 592, 'tokens': 9513}
trustworthy_history {'action': 'fold', 'raise_size': None, 'actual_increment': 0} historical A ranks [3, 4, 5] true/false 3 0
Round memo: B의 관찰 카드는 9C이며 내 카드는 미확인이다. B의 '10 이하' 주장은 과거 공개 결과 세 건에서 사실이었으므로 이번에도 참고하여 추가 지출 없이 폴드할 계획이다.
Long memo: B가 '네 카드는 10 이하야'라고 말한 과거 세 건에서 공개된 내 카드는 각각 3S·4S·5S로 주장과 일치했다. 같은 주장은 신뢰 근거로 참고하되 현재 카드의 확정 정보로 취급하지 않는다.
no_information {'action': 'call', 'raise_size': None, 'actual_increment': 0} historical A ranks [] true/false 0 0
Round memo: 관찰된 B의 카드는 9C이며, B의 '내 카드가 10 이하'라는 발언은 검증되지 않은 주장이다. 내 카드는 미확인이므로 주장을 확정 정보로 취급하지 않고, 콜 비용 4와 현재 팟 8을 고려해 콜할 잠정 계획이다.
Long memo: 
deceptive_history {'action': 'raise', 'raise_size': 'large', 'actual_increment': 4} historical A rank

In [2]:
from pathlib import Path
from collections import Counter, defaultdict
import json, statistics, math
root = next(p for p in [Path.cwd(), *Path.cwd().parents] if (p / 'experiments/results').is_dir())
path = root / 'experiments/results/20261021T090948-b54b5052'
def read(name):
    return [json.loads(l) for l in (path / (name + '.jsonl')).read_text().splitlines() if l.strip()]
trials, calls = read('trials'), read('model_calls')
manifest = json.loads((path / 'manifest.json').read_text())
requests = [c for c in calls if c['event'] == 'request']
responses = [c for c in calls if c['event'] == 'response']
assert len(trials) == len(requests) == len(responses) == 30
assert all(t['valid'] for t in trials)
assert all(r['usage']['actual_model'] == 'gpt-6.1-sol' for r in responses)
assert all(t['game_config']['openai_reasoning_effort'] == 'low' for t in trials)
assert manifest['source_hashes']['agents/prompts/poker_action_selection.txt'] == manifest['source_hashes']['agents/prompts/poker_action_selection_standard.txt']
assert len({q['system'] for q in requests}) == 1
assert len({json.dumps(q['schema'], sort_keys=True) for q in requests}) == 1
lookup = {t['trial_id']: t for t in trials}
inputs, histories, by_case = defaultdict(list), defaultdict(set), defaultdict(dict)
for q in requests:
    user = json.loads(q['user'])
    t = lookup[q['trial_id']]
    history = user.pop('past_experiences')
    inputs[t['case_id']].append(user)
    histories[t['condition']].add(json.dumps(history, sort_keys=True))
    assert user['force_bet'] and user['observation']['allowed_bets'] == ['call', 'raise', 'fold']
    assert 'A' not in user['observation']['visible_cards']
    assert user['observation']['visible_cards']['B'] == t['deal']['B']
    public = json.loads((path / 'data' / t['trial_id'] / 'experiences' / (t['game_id'] + '.json')).read_text())
    for match in history['matches']:
        parent = next(row for row in public if row['id'] == match['metadata']['experience_id'])
        assert match['round_context']['result'] == parent['result']
        assert json.loads(match['text']) in parent['events']
assert len({t['input_hash'] for t in trials}) == 10
assert all(all(u == group[0] for u in group) for group in inputs.values())
assert all(len(group) == 1 for group in histories.values())
assert all(not any(r['memo_inputs'].values()) for r in read('contexts') if 'memo_inputs' in r)
for t in trials:
    assert all(r['status'] == 'accepted' for r in t['applied'])
    assert t['decision']['memory_updates'] == t['saved_memos']
    base = path / 'data' / t['trial_id'] / 'memos/A'
    for scope, text in t['saved_memos'].items():
        file = base / 'opponents.txt' if scope == 'long' else base / t['game_id'] / 'game.txt' if scope == 'game' else base / t['game_id'] / 'rounds/000001.txt'
        assert file.read_text() == text
    by_case[t['case_id']][t['condition']] = t
print('Verified 30/30 accepted decisions; 10 distinct observations; only history differs within each deal.')
print('Actual models:', Counter(r['usage']['actual_model'] for r in responses))
print('Seconds:', round(manifest['total_duration_s'], 3), 'tokens:', {k: sum(r['usage'][k] for r in responses) for k in ('input_tokens', 'output_tokens', 'tokens')})
for condition in ('no_information', 'trustworthy_history', 'deceptive_history'):
    print(condition, dict(Counter(t['betting_signature']['action'] for t in trials if t['condition'] == condition)))
for a, b in [('no_information', 'trustworthy_history'), ('no_information', 'deceptive_history'), ('trustworthy_history', 'deceptive_history')]:
    count = sum(group[a]['betting_signature']['action'] != group[b]['betting_signature']['action'] for group in by_case.values())
    print(a, 'vs', b, 'action changes:', count, '/10')
for case, group in sorted(by_case.items()):
    print(case, 'B', group['no_information']['deal']['B'], [(c, t['betting_signature']) for c, t in sorted(group.items())])
model_times = sorted(t['duration_s'] for t in read('timings') if t['kind'] == 'model')
print('Model seconds: mean', statistics.mean(model_times), 'median', statistics.median(model_times), 'p95', model_times[math.ceil(.95 * len(model_times))-1])
print('Manual source audit: all 20 history-arm long memos accurately state the historical A ranks and claim truth. 17 explicitly link history to current assessment/plan; 3 trustworthy cases (02/08/09) leave that weighting unreported.')
print('Limit: outputs support information-sensitive decisions across these 10 deals; no retrieval, future memo reuse, win-rate or optimality result.')


Verified 30/30 accepted decisions; 10 distinct observations; only history differs within each deal.
Actual models: Counter({'gpt-6.1-sol': 30})
Seconds: 206.036 tokens: {'input_tokens': 89210, 'output_tokens': 6135, 'tokens': 95345}
no_information {'call': 4, 'fold': 4, 'raise': 2}
trustworthy_history {'fold': 6, 'call': 3, 'raise': 1}
deceptive_history {'raise': 5, 'fold': 1, 'call': 4}
no_information vs trustworthy_history action changes: 3 /10
no_information vs deceptive_history action changes: 6 /10
trustworthy_history vs deceptive_history action changes: 8 /10
s4-TRUST-01 B {'rank': 9, 'suit': 'C'} [('deceptive_history', {'action': 'raise', 'raise_size': 'large', 'actual_increment': 4}), ('no_information', {'action': 'call', 'raise_size': None, 'actual_increment': 0}), ('trustworthy_history', {'action': 'fold', 'raise_size': None, 'actual_increment': 0})]
s4-TRUST-02 B {'rank': 2, 'suit': 'S'} [('deceptive_history', {'action': 'raise', 'raise_size': 'maximum', 'actual_increment': 